In [1]:
# 1. Import libraries
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix, classification_report

# 2. Load and preprocess data
data = pd.read_csv("NASA TurboFan Engine.csv")
data = data.sort_values(["unit", "cycle"])

features = [f"sensor{i}" for i in range(1, 22)]

# Split by engine unit
units = data["unit"].unique()
train_units, test_units = train_test_split(
    units, test_size=0.2, random_state=42
)

train_data = data[data["unit"].isin(train_units)].copy()
test_data = data[data["unit"].isin(test_units)].copy()

# Scale using healthy training readings only
scaler = MinMaxScaler()
scaler.fit(train_data.loc[train_data["Fault_Label"] == 0, features])

train_data[features] = scaler.transform(train_data[features])
test_data[features] = scaler.transform(test_data[features])

# 3. Create sequences of 20 readings
window_size = 20

def create_sequences(data, normal_only=False):
    X, y = [], []

    for unit in data["unit"].unique():
        engine = data[data["unit"] == unit]
        values = engine[features].values
        labels = engine["Fault_Label"].values

        for i in range(len(engine) - window_size + 1):
            window = values[i:i + window_size]
            label = int(np.any(labels[i:i + window_size] == 1))

            if normal_only and label == 1:
                continue

            X.append(window)
            y.append(label)

    return np.array(X), np.array(y)

X_train, _ = create_sequences(train_data, normal_only=True)
X_test, y_test = create_sequences(test_data)

# 4. Build LSTM Autoencoder
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(window_size, len(features))),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.RepeatVector(window_size),
    tf.keras.layers.LSTM(32, return_sequences=True),
    tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(len(features))
    )
])

# 5. Compile and train
model.compile(optimizer="adam", loss="mse", metrics=["mae"])

history = model.fit(
    X_train, X_train,
    epochs=20, batch_size=32,
    validation_split=0.1
)

# 6. Calculate reconstruction error and threshold
train_pred = model.predict(X_train)
train_error = np.mean((X_train - train_pred)**2, axis=(1, 2))

threshold = np.percentile(train_error, 95)
print("Threshold:", threshold)

# 7. Detect faults
test_pred = model.predict(X_test)
test_error = np.mean((X_test - test_pred)**2, axis=(1, 2))

y_pred = (test_error > threshold).astype(int)

# 8. Evaluate model
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

# 9. Visualize training loss
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

# 10. Visualize reconstruction errors
plt.plot(test_error, label="Reconstruction Error")
plt.axhline(threshold, color="red", linestyle="--", label="Threshold")
plt.xlabel("Test Windows")
plt.ylabel("Reconstruction Error")
plt.legend()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'NASA TurboFan Engine.csv'